# V1-S15 — F1 epistemic cascade (pre-registered PR3, computed ONCE)

This notebook produces the **once-only** F1 result: within a leaf topic, does evidence
*quality/maturity* **LEAD** research-volume growth (rigor precedes the surge) or **LAG** it
(attention precedes rigor)? The analysis protocol — tier map, qualifying filter, span/NaN/
differencing rules, CCF window (L=±5), fixed Granger lag 3 (`ssr_ftest`), BH-FDR at q<0.05,
the panel-Granger confirmation, and the binary holds rule (≥20% directional AND agreeing
panel) — was **locked before this compute** in `docs/preregistrations/PR3_epistemic_cascade.md`
(internal git-timestamped pre-registration; the PR3 commit precedes this results commit).

All statistics live in the **pure** module `scifield.findings.cascade`; this notebook only does
the parquet I/O and the `epistemic_extracted ⋈ novelty_semantic` join, then calls the module
verbatim per PR3 §§5–8. **$0, CPU-only, no network/GPU/DeepSeek.** Per PR3 §9 and the project's
anti-drift discipline, the result is reported **exactly as returned** — a null F1 is reported as
a null, with no re-tuning, no re-run-with-different-params, and no favorable-variant rescue.

**Universe (PR3 §4):** papers in `epistemic_extracted.parquet` (`pmid, study_design`) INNER-joined
to `novelty_semantic.parquet` (`pmid, topic_id, year`) on `pmid`, LEAF topics only
(`topic_id != -1`) — ≈ 69,339 papers across 149 leaf topics, of which **138 qualify** (the fixed
holds-fraction denominator). The robustness rerun substitutes `rct_share` for `mean_tier`
(PR3 §8.3); the primary verdict is the mean-tier verdict, and the RCT-share rerun cannot
convert a null primary into a hold.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.findings import cascade as C  # noqa: E402
from scifield.repro import record_run  # noqa: E402

# Pinned PR3 parameters (echoed for the verdict sidecar; these are the module defaults).
PR3_PARAMS = {
    "tier_map": dict(
        C.TIER_MAP
    ),  # RCT=4 / cohort=3 / case_control=2 / case_series=1; review/other=NaN
    "v_min": 30,
    "min_years": 8,
    "min_count": 5,
    "max_nan_frac": 0.25,
    "ccf_max_lag": 5,
    "granger_lag": 3,
    "fdr_q": 0.05,
    "holds_frac": 0.20,
    "panel_alpha": 0.05,
    "directional_rule": "exactly-one-direction FDR-significant",
    "sign_convention": "negative peak lag = quality LEADS volume; positive = quality LAGS",
}


def _load_parquet(path: Path, columns=None) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot run the analysis.")
        return None
    return pd.read_parquet(path, columns=columns)


print("matplotlib:", matplotlib.__version__, "| backend:", matplotlib.get_backend())
print("DATA       :", DATA)
print("FIGURES_DIR:", FIGURES_DIR)

matplotlib: 3.10.9 | backend: Agg
DATA       : /Users/samersalman/Desktop/SciField/data/v1
FIGURES_DIR: /Users/samersalman/Desktop/SciField/docs/figures


## 2. I/O + join — assemble the F1 universe (PR3 §4)

INNER-join `epistemic_extracted[pmid, study_design]` to `novelty_semantic[pmid, topic_id, year]`
on `pmid`; keep LEAF topics only (`topic_id != -1`). The `study_design` values are exactly
`RCT/cohort/case_control/case_series/review/other` and match the `TIER_MAP` keys — they pass
through unchanged. The module is pure: it receives this already-joined, already-leaf-filtered
tidy frame and does the rest.

In [2]:
EPISTEMIC = DATA / "epistemic_extracted.parquet"
NOVELTY = DATA / "novelty_semantic.parquet"

ep = _load_parquet(EPISTEMIC, columns=["pmid", "study_design"])
nov = _load_parquet(NOVELTY, columns=["pmid", "topic_id", "year"])
assert ep is not None and nov is not None, "F1 inputs missing — cannot proceed."

df = ep.merge(nov, on="pmid", how="inner")  # INNER join on pmid
n_joined = len(df)
df = df[df["topic_id"] != -1].copy()  # LEAF topics only
n_leaf = len(df)
n_leaf_topics = int(df["topic_id"].nunique())

designs = sorted(df["study_design"].dropna().unique().tolist())
print(f"epistemic rows = {len(ep):,} | novelty rows = {len(nov):,}")
print(f"INNER join     = {n_joined:,} rows")
print(f"leaf-filtered  = {n_leaf:,} papers across {n_leaf_topics} leaf topics (F1 universe)")
print(f"year span      = {int(df['year'].min())}–{int(df['year'].max())}")
print(f"study_design values = {designs}")
print(f"NaN study_design in universe = {int(df['study_design'].isna().sum())}")
# Sanity vs PR3 §4 pre-lock descriptive denominators (≈ 69,339 papers, 149 leaf topics).
assert set(designs) <= set(
    ["RCT", "cohort", "case_control", "case_series", "review", "other"]
), "unexpected study_design label — does not match TIER_MAP universe"

epistemic rows = 91,230 | novelty rows = 89,230
INNER join     = 91,230 rows
leaf-filtered  = 69,339 papers across 149 leaf topics (F1 universe)
year span      = 1995–2026
study_design values = ['RCT', 'case_control', 'case_series', 'cohort', 'other', 'review']
NaN study_design in universe = 0


## 3. The canonical analysis loop (PR3 §§5–8, implemented verbatim)

For a given quality column the loop: (1) builds per-`(topic, year)` series and the qualifying
topic list (the 138 denominator); (2) per qualifying topic prepares the two series; if
`status == "ok"` it runs bidirectional Granger and appends **both** p-values to a single pooled
vector (consistent `[p_qv, p_vq]` order, with an index map back to `(topic, direction)`), records
the differenced pair for the panel, and the CCF peak lag; if `status in {non_evaluable,
too_short}` the topic STAYS in the 138 denominator with `direction="none"` and is added to
neither the pool nor the panel. (3) BH-FDR is applied **once over the WHOLE pool** (all topics,
both directions) — never per-topic; each topic's `leads_sig`/`lags_sig` come from its two reject
flags and `classify_direction` sets its direction. (4) `panel_granger` runs on the differenced
pairs (evaluable topics only); (5) `decide_f1` returns the verdict.

In [3]:
def run_f1(frame: pd.DataFrame, *, quality_col: str) -> dict:
    """Execute the full pre-registered F1 loop for one quality series.

    Returns a dict with the verdict (`decide_f1` output), the per-topic results list,
    the panel p-values, the qualifying/non-evaluable counts, and the CCF peak-lag list.
    Calls the pure `cascade` helpers only — it never reimplements the statistics.
    """
    series = C.build_topic_year_series(frame)  # one row per (topic, year)
    topics = C.qualifying_topics(series)  # defaults 30/8/5 -> the 138 denominator
    n_qualifying = len(topics)

    pooled_pvals: list[float] = []  # ALL per-topic per-direction p's, one flat vector
    pool_index: list[tuple] = []  # parallel: pool_position -> (topic_id, direction)
    differenced: list[tuple] = []  # (Δquality, Δvolume) per evaluable topic, for the panel
    per_topic: list[dict] = []  # per-topic results table rows
    peak_lags: list[int] = []  # CCF peak lag per evaluable topic
    n_non_evaluable = 0
    n_too_short = 0

    for t in topics:
        topic_series = series[series.topic_id == t]
        quality, volume, status = C.prepare_series(topic_series, quality_col=quality_col)

        if status == "ok":
            p_qv, p_vq = C.granger_pair(quality, volume)
            # Append BOTH directions to the pool in a CONSISTENT order [p_qv, p_vq].
            pos_qv = len(pooled_pvals)
            pooled_pvals.append(p_qv)
            pool_index.append((int(t), "q_to_v"))
            pos_vq = len(pooled_pvals)
            pooled_pvals.append(p_vq)
            pool_index.append((int(t), "v_to_q"))
            # Panel input is the pre-differenced pair (evaluable topics only).
            differenced.append((np.diff(quality.to_numpy()), np.diff(volume.to_numpy())))
            peak_lag, peak_corr, _ccf = C.cross_correlation(quality, volume)
            peak_lags.append(int(peak_lag))
            per_topic.append(
                {
                    "topic_id": int(t),
                    "status": status,
                    "peak_lag": int(peak_lag),
                    "peak_corr": float(peak_corr),
                    "p_q_to_v": float(p_qv),
                    "p_v_to_q": float(p_vq),
                    "leads_sig": False,  # filled after pooled FDR
                    "lags_sig": False,
                    "direction": "none",
                    "_pos_qv": pos_qv,
                    "_pos_vq": pos_vq,
                }
            )
        else:
            # non_evaluable / too_short STAY in the 138 denominator as direction='none';
            # NOT added to the pool or the panel (conservative — PR3 §5).
            if status == "non_evaluable":
                n_non_evaluable += 1
            elif status == "too_short":
                n_too_short += 1
            per_topic.append(
                {
                    "topic_id": int(t),
                    "status": status,
                    "peak_lag": pd.NA,
                    "peak_corr": float("nan"),
                    "p_q_to_v": float("nan"),
                    "p_v_to_q": float("nan"),
                    "leads_sig": False,
                    "lags_sig": False,
                    "direction": "none",
                    "_pos_qv": None,
                    "_pos_vq": None,
                }
            )

    # THE multiple-comparison correction: BH-FDR over the WHOLE pool (all topics, both
    # directions). NaN p's come back not-rejected and do not inflate m.
    reject = C.bh_fdr(np.array(pooled_pvals, dtype="float64"))
    for r in per_topic:
        if r["_pos_qv"] is not None:
            leads_sig = bool(reject[r["_pos_qv"]])  # its q_to_v reject flag
            lags_sig = bool(reject[r["_pos_vq"]])  # its v_to_q reject flag
            r["leads_sig"] = leads_sig
            r["lags_sig"] = lags_sig
            r["direction"] = C.classify_direction(leads_sig=leads_sig, lags_sig=lags_sig)

    panel = C.panel_granger(differenced)  # (p_q_to_v, p_v_to_q) block-F on the panel
    verdict = C.decide_f1(per_topic, panel, n_qualifying=n_qualifying)

    # Drop the private pool-position helpers from the persisted table.
    results_df = pd.DataFrame(
        [{k: v for k, v in r.items() if not k.startswith("_")} for r in per_topic]
    )
    return {
        "verdict": verdict,
        "results_df": results_df,
        "panel": (float(panel[0]), float(panel[1])),
        "n_qualifying": int(n_qualifying),
        "n_non_evaluable": int(n_non_evaluable),
        "n_too_short": int(n_too_short),
        "n_evaluable": int(len(differenced)),
        "pool_size": int(len(pooled_pvals)),
        "peak_lags": peak_lags,
        "series": series,
    }

### 3.1 PRIMARY run — mean-tier quality series

In [4]:
primary = run_f1(df, quality_col="mean_tier")
vP = primary["verdict"]
series = primary["series"]  # reused for the figure exemplar trajectories

print("=== PRIMARY (mean-tier) ===")
print(f"n_qualifying      = {primary['n_qualifying']}  (PR3 fixed denominator = 138)")
print(f"n_non_evaluable   = {primary['n_non_evaluable']}   (>25% NaN-tier; kept in denom)")
print(f"n_too_short       = {primary['n_too_short']}")
print(f"n_evaluable       = {primary['n_evaluable']}   (pooled p-values = {primary['pool_size']})")
print(
    f"direction split   : lead={vP['n_lead']}  lag={vP['n_lag']}  "
    f"coupled={vP['n_coupled']}  none={vP['n_none']}"
)
print(
    f"n_directional     = {vP['n_directional']}   frac = {vP['frac_directional']:.4f}  "
    f"(holds bar = {vP['frac_threshold']:.2f} → ≥ 28/138)"
)
print(f"dominant_direction= {vP['dominant_direction']}   panel_agrees = {vP['panel_agrees']}")
print(
    f"panel p: quality→volume = {vP['panel_p_quality_leads']:.4g}  "
    f"volume→quality = {vP['panel_p_quality_lags']:.4g}"
)
print(f"\nF1 HOLDS (primary) = {vP['holds']}")

=== PRIMARY (mean-tier) ===
n_qualifying      = 138  (PR3 fixed denominator = 138)
n_non_evaluable   = 10   (>25% NaN-tier; kept in denom)
n_too_short       = 0
n_evaluable       = 128   (pooled p-values = 256)
direction split   : lead=0  lag=0  coupled=0  none=138
n_directional     = 0   frac = 0.0000  (holds bar = 0.20 → ≥ 28/138)
dominant_direction= none   panel_agrees = False
panel p: quality→volume = 0.2771  volume→quality = 0.7393

F1 HOLDS (primary) = False


### 3.2 ROBUSTNESS rerun — RCT-share quality series (PR3 §8.3)

Identical loop with `quality_col="rct_share"`. `rct_share` has no structural NaNs (its
denominator is the full topic-year count), so no topic should be flagged non-evaluable here.
The RCT-share verdict characterizes robustness and **cannot** convert a null primary into a hold
(PR3 §9).

In [5]:
rct = run_f1(df, quality_col="rct_share")
vR = rct["verdict"]

print("=== ROBUSTNESS (rct_share) ===")
print(f"n_qualifying      = {rct['n_qualifying']}")
print(f"n_non_evaluable   = {rct['n_non_evaluable']}   (expected 0; gap-free series)")
print(f"n_evaluable       = {rct['n_evaluable']}   (pooled p-values = {rct['pool_size']})")
print(
    f"direction split   : lead={vR['n_lead']}  lag={vR['n_lag']}  "
    f"coupled={vR['n_coupled']}  none={vR['n_none']}"
)
print(f"n_directional     = {vR['n_directional']}   frac = {vR['frac_directional']:.4f}")
print(f"dominant_direction= {vR['dominant_direction']}   panel_agrees = {vR['panel_agrees']}")
print(
    f"panel p: quality→volume = {vR['panel_p_quality_leads']:.4g}  "
    f"volume→quality = {vR['panel_p_quality_lags']:.4g}"
)
print(f"\nF1 HOLDS (rct_share) = {vR['holds']}")

AGREE = bool(vP["holds"] == vR["holds"])
print(
    f"\nPrimary and RCT-share verdicts AGREE = {AGREE}  "
    f"(primary holds={vP['holds']}, rct_share holds={vR['holds']})"
)

=== ROBUSTNESS (rct_share) ===
n_qualifying      = 138
n_non_evaluable   = 0   (expected 0; gap-free series)
n_evaluable       = 138   (pooled p-values = 276)
direction split   : lead=0  lag=0  coupled=0  none=138
n_directional     = 0   frac = 0.0000
dominant_direction= none   panel_agrees = False
panel p: quality→volume = 0.9502  volume→quality = 0.9562

F1 HOLDS (rct_share) = False

Primary and RCT-share verdicts AGREE = True  (primary holds=False, rct_share holds=False)


### 3.3 The F1 verdict, stated plainly

In [6]:
F1_LABEL = "HOLDS" if vP["holds"] else "NULL"
print("=" * 72)
print(f"F1 VERDICT (primary, mean-tier) : F1 {F1_LABEL}")
print("=" * 72)
if not vP["holds"]:
    print(
        "F1 is a NULL finding: there is NO FDR-significant directional quality↔volume\n"
        "cascade in ≥20% of qualifying topics confirmed by an agreeing panel test.\n"
        f"  • directional topics = {vP['n_directional']} / {primary['n_qualifying']} "
        f"(frac {vP['frac_directional']:.4f} < {vP['frac_threshold']:.2f}; criterion (a) FAILS)\n"
        f"  • panel non-significant in both directions — criterion (b) cannot be met\n"
        "  • RCT-share robustness rerun AGREES (also null); a favorable robustness verdict\n"
        "    could not rescue it in any case (PR3 §9).\n"
        "Reported faithfully as a null — NOT re-tuned, NOT softened, NOT inflated."
    )
else:
    print(
        f"F1 HOLDS: {vP['n_directional']}/{primary['n_qualifying']} topics directional "
        f"(frac {vP['frac_directional']:.4f} ≥ {vP['frac_threshold']:.2f}), dominant "
        f"{vP['dominant_direction']}, panel agrees."
    )

F1 VERDICT (primary, mean-tier) : F1 NULL
F1 is a NULL finding: there is NO FDR-significant directional quality↔volume
cascade in ≥20% of qualifying topics confirmed by an agreeing panel test.
  • directional topics = 0 / 138 (frac 0.0000 < 0.20; criterion (a) FAILS)
  • panel non-significant in both directions — criterion (b) cannot be met
  • RCT-share robustness rerun AGREES (also null); a favorable robustness verdict
    could not rescue it in any case (PR3 §9).
Reported faithfully as a null — NOT re-tuned, NOT softened, NOT inflated.


## 4. Figure F1 — `docs/figures/F1_epistemic_cascade.png`

Three panels: **(a)** exemplar quality/volume LEVEL trajectories for the qualifying topics with
the clearest directional Granger signal (smallest per-direction p); since **no** topic survives
BH-FDR, the panel annotates the *candidate* direction and states it is not FDR-significant.
**(b)** the CCF peak-lag distribution across qualifying topics (negative lag = quality leads).
**(c)** the per-topic direction split (lead / lag / coupled / none). The caption records the
null verdict and that F1 is pre-registered (PR3).

In [7]:
# Readable topic labels from the hierarchy's top_words (first 3 words).
th = _load_parquet(DATA / "topic_hierarchy.parquet", columns=["topic_id", "top_words"])
TOPIC_WORDS = {}
if th is not None:
    for row in th.itertuples():
        words = list(row.top_words)
        TOPIC_WORDS[int(row.topic_id)] = ", ".join(str(w) for w in words[:3])


def _candidate_direction(p_qv: float, p_vq: float) -> tuple[str, float]:
    """Smaller per-direction Granger p → candidate (not necessarily FDR-significant) direction."""
    pqv = np.inf if not np.isfinite(p_qv) else p_qv
    pvq = np.inf if not np.isfinite(p_vq) else p_vq
    if pqv <= pvq:
        return "quality leads (q→v)", pqv
    return "quality lags (v→q)", pvq


# Exemplars: evaluable qualifying topics ranked by smallest per-direction Granger p.
res_ok = primary["results_df"][primary["results_df"]["status"] == "ok"].copy()
res_ok["p_min"] = res_ok[["p_q_to_v", "p_v_to_q"]].min(axis=1)
exemplar_ids = res_ok.sort_values("p_min").head(3)["topic_id"].astype(int).tolist()
print("exemplar topics (smallest per-direction Granger p):", exemplar_ids)

exemplar topics (smallest per-direction Granger p): [110, 76, 55]


In [8]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.6))
ax_a0, ax_a1, ax_b, ax_c = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]
exemplar_axes = [ax_a0, ax_a1]  # show the 2 clearest as twin-axis level trajectories

# --- (a) exemplar level trajectories (quality mean-tier vs volume), twin y-axis ---
for ax, tid in zip(exemplar_axes, exemplar_ids[:2], strict=False):
    ts = series[series.topic_id == tid].sort_values("year")
    quality, volume, status = C.prepare_series(ts, quality_col="mean_tier")
    yrs = quality.index.to_numpy()
    rrow = res_ok[res_ok["topic_id"] == tid].iloc[0]
    cand, pmin = _candidate_direction(rrow["p_q_to_v"], rrow["p_v_to_q"])
    ax.plot(
        yrs,
        quality.to_numpy(),
        color="#ea4335",
        marker="o",
        ms=3,
        lw=1.4,
        label="evidence quality (mean tier)",
    )
    ax.set_ylabel("mean evidence tier", color="#ea4335", fontsize=8)
    ax.tick_params(axis="y", labelcolor="#ea4335", labelsize=7)
    ax2 = ax.twinx()
    ax2.plot(
        yrs,
        volume.to_numpy(),
        color="#4285f4",
        marker="s",
        ms=3,
        lw=1.4,
        label="research volume (papers/yr)",
    )
    ax2.set_ylabel("volume (papers/yr)", color="#4285f4", fontsize=8)
    ax2.tick_params(axis="y", labelcolor="#4285f4", labelsize=7)
    ax.tick_params(axis="x", labelsize=7)
    label = TOPIC_WORDS.get(tid, f"topic {tid}")
    peak_lag = int(rrow["peak_lag"]) if pd.notna(rrow["peak_lag"]) else 0
    ax.set_title(
        f"(a) topic {tid}: {label}\ncandidate: {cand}  (Granger p={pmin:.3f}, "
        f"NOT FDR-sig; CCF peak lag={peak_lag:+d})",
        fontsize=8,
    )
    lines, labs = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(lines + l2, labs + lb2, fontsize=6, loc="upper left")

# --- (b) CCF peak-lag distribution across qualifying (evaluable) topics ---
peak_lags = np.array(primary["peak_lags"], dtype=int)
bins = np.arange(-PR3_PARAMS["ccf_max_lag"] - 0.5, PR3_PARAMS["ccf_max_lag"] + 1.5, 1.0)
ax_b.hist(peak_lags, bins=bins, color="#9aa0a6", edgecolor="white")
ax_b.axvline(0, ls=":", c="#444444", lw=1)
ax_b.set_xlabel("CCF peak lag (years)")
ax_b.set_ylabel("# qualifying topics")
ax_b.set_title(
    f"(b) CCF peak-lag distribution (n={peak_lags.size} evaluable)\n"
    "negative = quality leads | positive = quality lags",
    fontsize=8.5,
)
ax_b.text(
    0.02,
    0.95,
    f"median lag = {int(np.median(peak_lags)):+d}",
    transform=ax_b.transAxes,
    fontsize=7,
    va="top",
)

# --- (c) per-topic direction split ---
split_labels = ["lead", "lag", "coupled", "none"]
split_counts = [vP["n_lead"], vP["n_lag"], vP["n_coupled"], vP["n_none"]]
split_colors = ["#ea4335", "#fbbc04", "#a142f4", "#9aa0a6"]
bars = ax_c.bar(split_labels, split_counts, color=split_colors)
ax_c.bar_label(bars, fontsize=8)
ax_c.set_ylabel("# qualifying topics")
ax_c.set_title(
    f"(c) Per-topic direction split (denominator = {primary['n_qualifying']})\n"
    f"directional = lead+lag = {vP['n_directional']} "
    f"(frac {vP['frac_directional']:.3f} < {vP['frac_threshold']:.2f} holds bar)",
    fontsize=8.5,
)
ax_c.set_ylim(0, max(split_counts) * 1.12)

verdict_txt = "F1 HOLDS" if vP["holds"] else "F1 = NULL"
fig.suptitle(
    f"F1 — Epistemic cascade (quality↔volume lead/lag), pre-registered PR3: {verdict_txt}  |  "
    f"primary={'holds' if vP['holds'] else 'null'}, rct_share={'holds' if vR['holds'] else 'null'} "
    f"(agree={AGREE})",
    fontsize=11,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
F1_FIG = FIGURES_DIR / "F1_epistemic_cascade.png"
fig.savefig(F1_FIG, dpi=DPI)
plt.close(fig)
print("wrote", F1_FIG)

wrote /Users/samersalman/Desktop/SciField/docs/figures/F1_epistemic_cascade.png


In [9]:
# Enforce the < 1 MB figure budget.
sz = F1_FIG.stat().st_size
print(f"F1 size = {sz / 1024:.1f} KB  (dpi={DPI})")
assert sz < 1_000_000, f"F1 too large: {sz} bytes"
print("F1 size assertion PASS — under 1 MB")

F1 size = 173.7 KB  (dpi=120)
F1 size assertion PASS — under 1 MB


In [10]:
# Provenance sidecar for the figure (mirrors notebook 10's F3 pattern).
fig_sidecar = record_run(
    artifact_path=F1_FIG,
    inputs={
        "epistemic": DATA / "epistemic_extracted.parquet",
        "novelty": DATA / "novelty_semantic.parquet",
    },
    config={
        "figure": "F1_epistemic_cascade",
        "session": "V1-S15",
        "prereg": "PR3_epistemic_cascade",
        "dpi": DPI,
        "n_universe_papers": int(n_leaf),
        "n_leaf_topics": int(n_leaf_topics),
        "n_qualifying": int(primary["n_qualifying"]),
        "n_non_evaluable_primary": int(primary["n_non_evaluable"]),
        "primary_holds": bool(vP["holds"]),
        "rct_share_holds": bool(vR["holds"]),
        "verdicts_agree": AGREE,
        "n_directional_primary": int(vP["n_directional"]),
        "frac_directional_primary": float(vP["frac_directional"]),
        "dominant_direction_primary": vP["dominant_direction"],
        "panel_p_quality_leads_primary": float(vP["panel_p_quality_leads"]),
        "panel_p_quality_lags_primary": float(vP["panel_p_quality_lags"]),
        "exemplar_topics": exemplar_ids,
        "params": PR3_PARAMS,
    },
)
print("recorded figure sidecar:", fig_sidecar)
print("F1 exists:", F1_FIG.exists(), "| KB:", f"{F1_FIG.stat().st_size / 1024:.1f}")

recorded figure sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F1_epistemic_cascade.png.run.json
F1 exists: True | KB: 173.7


## 5. Persist results

Write the **primary** per-topic results table (`f1_cascade_results.parquet`) and the
machine-readable verdict (`f1_cascade_verdict.json`, primary + RCT-share + agreement +
denominators + pinned params), each with a `record_run` sidecar.

In [11]:
# Per-topic results table (PRIMARY mean-tier run) — the contract's column set.
RESULT_COLS = [
    "topic_id",
    "status",
    "peak_lag",
    "peak_corr",
    "p_q_to_v",
    "p_v_to_q",
    "leads_sig",
    "lags_sig",
    "direction",
]
results_df = primary["results_df"][RESULT_COLS].copy()
# peak_lag is nullable (NA for non-evaluable/too-short topics).
results_df["peak_lag"] = results_df["peak_lag"].astype("Int64")
results_df["topic_id"] = results_df["topic_id"].astype("int64")

RESULTS_PARQUET = DATA / "f1_cascade_results.parquet"
results_df.to_parquet(RESULTS_PARQUET, index=False)
print("wrote", RESULTS_PARQUET, "| rows =", len(results_df))

results_sidecar = record_run(
    artifact_path=RESULTS_PARQUET,
    inputs={
        "epistemic": DATA / "epistemic_extracted.parquet",
        "novelty": DATA / "novelty_semantic.parquet",
    },
    config={
        "artifact": "f1_cascade_results",
        "session": "V1-S15",
        "prereg": "PR3_epistemic_cascade",
        "quality_series": "mean_tier (PRIMARY)",
        "n_rows": int(len(results_df)),
        "n_qualifying": int(primary["n_qualifying"]),
        "n_evaluable": int(primary["n_evaluable"]),
        "n_non_evaluable": int(primary["n_non_evaluable"]),
        "n_too_short": int(primary["n_too_short"]),
        "pool_size": int(primary["pool_size"]),
        "params": PR3_PARAMS,
    },
)
print("recorded results sidecar:", results_sidecar)
results_df.head()

wrote /Users/samersalman/Desktop/SciField/data/v1/f1_cascade_results.parquet | rows = 138
recorded results sidecar: /Users/samersalman/Desktop/SciField/data/v1/f1_cascade_results.parquet.run.json


,topic_id,status,peak_lag,peak_corr,p_q_to_v,p_v_to_q,leads_sig,lags_sig,direction
0,0,ok,-4,0.492745,0.396630,0.138761,False,False,none
1,1,ok,1,0.567338,0.643884,0.196028,False,False,none
2,2,ok,-5,0.496177,0.358326,0.556054,False,False,none
3,3,ok,3,0.525283,0.543348,0.072517,False,False,none
4,4,ok,2,-0.471867,0.954473,0.154099,False,False,none


In [12]:
# Machine-readable verdict: primary + rct_share + agreement + denominators + params.
VERDICT_JSON = DATA / "f1_cascade_verdict.json"
verdict_payload = {
    "primary": vP,
    "rct_share": vR,
    "agree": AGREE,
    "n_qualifying": int(primary["n_qualifying"]),
    "n_non_evaluable": int(primary["n_non_evaluable"]),
    "n_non_evaluable_rct_share": int(rct["n_non_evaluable"]),
    "n_too_short": int(primary["n_too_short"]),
    "n_evaluable_primary": int(primary["n_evaluable"]),
    "n_evaluable_rct_share": int(rct["n_evaluable"]),
    "panel_primary": {
        "p_quality_leads": primary["panel"][0],
        "p_quality_lags": primary["panel"][1],
    },
    "panel_rct_share": {
        "p_quality_leads": rct["panel"][0],
        "p_quality_lags": rct["panel"][1],
    },
    "f1_verdict": "HOLDS" if vP["holds"] else "NULL",
    "n_universe_papers": int(n_leaf),
    "n_leaf_topics": int(n_leaf_topics),
    "params": PR3_PARAMS,
}
VERDICT_JSON.write_text(json.dumps(verdict_payload, indent=2, sort_keys=False))
print("wrote", VERDICT_JSON)

verdict_sidecar = record_run(
    artifact_path=VERDICT_JSON,
    inputs={
        "epistemic": DATA / "epistemic_extracted.parquet",
        "novelty": DATA / "novelty_semantic.parquet",
    },
    config={
        "artifact": "f1_cascade_verdict",
        "session": "V1-S15",
        "prereg": "PR3_epistemic_cascade",
        "f1_verdict": "HOLDS" if vP["holds"] else "NULL",
        "primary_holds": bool(vP["holds"]),
        "rct_share_holds": bool(vR["holds"]),
        "verdicts_agree": AGREE,
        "params": PR3_PARAMS,
    },
)
print("recorded verdict sidecar:", verdict_sidecar)
print(json.dumps(verdict_payload["primary"], indent=2))

wrote /Users/samersalman/Desktop/SciField/data/v1/f1_cascade_verdict.json
recorded verdict sidecar: /Users/samersalman/Desktop/SciField/data/v1/f1_cascade_verdict.json.run.json
{
  "holds": false,
  "n_directional": 0,
  "frac_directional": 0.0,
  "dominant_direction": "none",
  "panel_agrees": false,
  "n_lead": 0,
  "n_lag": 0,
  "n_coupled": 0,
  "n_none": 138,
  "panel_p_quality_leads": 0.27712455387307633,
  "panel_p_quality_lags": 0.7392799167397095,
  "frac_threshold": 0.2
}


## 6. Verify artifacts

In [13]:
artifacts = [
    F1_FIG,
    F1_FIG.with_suffix(F1_FIG.suffix + ".run.json"),
    RESULTS_PARQUET,
    RESULTS_PARQUET.with_suffix(RESULTS_PARQUET.suffix + ".run.json"),
    VERDICT_JSON,
    VERDICT_JSON.with_suffix(VERDICT_JSON.suffix + ".run.json"),
]
for a in artifacts:
    ok = a.exists()
    kb = f"{a.stat().st_size / 1024:.1f} KB" if ok else "MISSING"
    print(f"  [{'ok' if ok else 'XX'}] {a.name:<48} {kb}")
    assert ok, f"artifact missing: {a}"
assert F1_FIG.stat().st_size < 1_000_000, "figure exceeds 1 MB"
print("\nAll F1 artifacts + sidecars present; figure under 1 MB.")
print(
    f"\nFINAL: F1 {'HOLDS' if vP['holds'] else 'is a NULL finding'} "
    f"(primary mean-tier); RCT-share {'holds' if vR['holds'] else 'null'}; agree={AGREE}."
)

  [ok] F1_epistemic_cascade.png                         173.7 KB
  [ok] F1_epistemic_cascade.png.run.json                1.6 KB
  [ok] f1_cascade_results.parquet                       9.5 KB
  [ok] f1_cascade_results.parquet.run.json              1.3 KB
  [ok] f1_cascade_verdict.json                          1.6 KB
  [ok] f1_cascade_verdict.json.run.json                 1.3 KB

All F1 artifacts + sidecars present; figure under 1 MB.

FINAL: F1 is a NULL finding (primary mean-tier); RCT-share null; agree=True.
